In [ ]:
import re
import nltk
import numpy as np
from tqdm import tqdm

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
import pandas as pd
messages=pd.read_csv('smsspamcollection/SMSSpamCollection',
                    sep='\t',names=["label","message"])

In [ ]:
corpus = []

for i in range(len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()

    # No stopword removal
    review = [lemmatizer.lemmatize(word) for word in review]

    review = ' '.join(review)
    corpus.append(review)

In [ ]:
corpus

In [ ]:
words = []

for sentence in corpus:
    sent_tokens = sent_tokenize(sentence)

    for sent in sent_tokens:
        words.append(simple_preprocess(sent))

In [ ]:
words

In [ ]:
model = Word2Vec(
    sentences=words,
    vector_size=100,
    window=5,
    min_count=1,
    epochs=100
)

In [ ]:
model.wv.index_to_key

In [ ]:
model.corpus_count

In [ ]:
model.wv.similar_by_word("good")

In [ ]:
model.wv.most_similar("good")

In [ ]:
model.wv["good"]

In [ ]:
model.wv["good"].shape

In [ ]:
def avg_word2vec(doc, model):
    return np.mean(
        [model.wv[word]
         for word in doc
         if word in model.wv.index_to_key],
        axis=0
    )

In [ ]:
def avg_word2vec(doc, model):
    vectors = [
        model.wv[word]
        for word in doc
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
X = []

for sentence in tqdm(words):
    X.append(avg_word2vec(sentence, model))

In [ ]:
X_new = np.array(X)

In [ ]:
X_new.shape

In [ ]:
X_new[0]

In [ ]:
X_new[0].shape

In [ ]:
y = messages['label']
y=y.iloc[:,0].values

In [ ]:
messages.shape

In [ ]:
for length, cleaned_msg, original_msg in zip(
    map(len, corpus),
    corpus,
    messages["message"]
):
    if length < 1:
        print(original_msg)